In [1]:
# If needed:
# !pip install numpy pandas scikit-learn


import warnings
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

In [2]:
books=pd.read_csv(r"D:\Data Science_bat\Recommendation_system\data\Books.csv")
rating=pd.read_csv(r"D:\Data Science_bat\Recommendation_system\data\Ratings.csv")
users=pd.read_csv(r"D:\Data Science_bat\Recommendation_system\data\Users.csv")


In [3]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [4]:
rating.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [5]:
users

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN
...,...,...,...
278853,278854,"portland, oregon, usa",NaN
278854,278855,"tacoma, washington, united kingdom",50.0
278855,278856,"brampton, ontario, canada",NaN
278856,278857,"knoxville, tennessee, usa",NaN


In [6]:
def clean_data(books, ratings, users):
    books.columns = [c.strip() for c in books.columns]
    books = books.rename(columns={
        "Book-Title": "title",
        "Book-Author": "author",
        "Year-Of-Publication": "year",
        "Publisher": "publisher",
    })

    books["year"] = pd.to_numeric(books["year"], errors="coerce")
    books = books.dropna(subset=["ISBN", "title"]).drop_duplicates("ISBN")

    ratings = ratings.rename(columns={
        "User-ID": "user_id",
        "Book-Rating": "rating",
    })
    ratings = ratings.dropna()
    ratings = ratings[ratings["rating"] > 0]

    users = users.rename(columns={"User-ID": "user_id"})
    users = users.dropna(subset=["user_id"])

    return books, ratings, users

In [7]:
def build_pivot(books, ratings, min_user_ratings=10, min_book_ratings=50):
    merged = ratings.merge(books[["ISBN","title"]], on="ISBN")

    active_users = (
        merged.groupby("user_id")["rating"].count() >= min_user_ratings
    )
    merged = merged[merged["user_id"].isin(active_users[active_users].index)]

    popular_books = (
        merged.groupby("title")["rating"].count() >= min_book_ratings
    )
    merged = merged[merged["title"].isin(popular_books[popular_books].index)]

    pivot = merged.pivot_table(index="title", columns="user_id", values="rating")
    pivot.fillna(0, inplace=True)

    print("Pivot shape:", pivot.shape)
    return pivot

In [8]:
def build_similarity(pivot):
    similarity = cosine_similarity(pivot)
    print("Similarity shape:", similarity.shape)
    return similarity

In [9]:
def similar_books(book_title, pivot, similarity, books, top_n=6):
    if book_title not in pivot.index:
        print("Book not found.")
        return []

    idx = list(pivot.index).index(book_title)
    scores = sorted(
        enumerate(similarity[idx]),
        key=lambda x: x[1],
        reverse=True
    )[1:top_n+1]

    recommendations = []
    for i, score in scores:
        title = pivot.index[i]
        meta = books[books["title"] == title]

        recommendations.append({
            "title": title,
            "author": meta["author"].values[0],
            "score": round(score, 3)
        })

    return recommendations

In [11]:
def cluster_users(pivot, n_clusters=5):
    user_matrix = pivot.T

    scaler = StandardScaler()
    X = scaler.fit_transform(user_matrix)

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=15
    )

    labels = kmeans.fit_predict(X)
    user_cluster = dict(zip(user_matrix.index, labels))

    return labels, user_cluster, user_matrix

In [12]:
def recommend_for_user(user_id, user_matrix, user_cluster, books, ratings, top_n=8):

    if user_id not in user_matrix.index:
        print("User not in dataset.")
        return []

    cluster_id = user_cluster[user_id]
    peers = [u for u, c in user_cluster.items()
             if c == cluster_id and u != user_id]

    target_vec = user_matrix.loc[user_id].values.reshape(1,-1)
    peer_matrix = user_matrix.loc[peers]

    sims = cosine_similarity(target_vec, peer_matrix)[0]

    read_books = set(user_matrix.loc[user_id][user_matrix.loc[user_id] > 0].index)
    unread = [b for b in user_matrix.columns if b not in read_books]

    results = []

    for title in unread:
        peer_ratings = peer_matrix[title].values
        mask = peer_ratings > 0

        if mask.sum() == 0:
            continue

        weighted_score = np.dot(sims[mask], peer_ratings[mask]) / (
            np.abs(sims[mask]).sum() + 1e-9
        )

        results.append((title, weighted_score))

    results.sort(key=lambda x: x[1], reverse=True)

    return results[:top_n]

In [85]:
from IPython.display import HTML, display


image_path = "https://images.amazon.com/images/P/0380018179.0"
display(HTML(f'<img src="{image_path}" style="max-width:700px; border:1px solid #ccc;">'))


In [13]:
books, rating, users = clean_data(books, rating, users)

pivot = build_pivot(books, rating)
similarity = build_similarity(pivot)

labels, user_cluster, user_matrix = cluster_users(pivot)

Pivot shape: (315, 5026)
Similarity shape: (315, 315)


In [14]:
similar_books("The Notebook", pivot, similarity, books)

[{'title': 'Message in a Bottle',
  'author': 'Nicholas Sparks',
  'score': np.float64(0.209)},
 {'title': 'A Walk to Remember',
  'author': 'Nicholas Sparks',
  'score': np.float64(0.204)},
 {'title': "Suzanne's Diary for Nicholas",
  'author': 'James Patterson',
  'score': np.float64(0.186)},
 {'title': 'The Rescue',
  'author': 'Nicholas Sparks',
  'score': np.float64(0.184)},
 {'title': 'The Wedding',
  'author': 'Nicholas Sparks',
  'score': np.float64(0.176)},
 {'title': 'The Five People You Meet in Heaven',
  'author': 'Mitch Albom',
  'score': np.float64(0.139)}]

In [15]:
recommend_for_user(242, user_matrix, user_cluster, books, rating)

[('Summer Sisters', np.float64(9.999999985857865)),
 ('Hard Eight : A Stephanie Plum Novel (A Stephanie Plum Novel)',
  np.float64(9.999999983236945)),
 ('G Is for Gumshoe (Kinsey Millhone Mysteries (Paperback))',
  np.float64(9.999999977218428)),
 ('The Little Prince', np.float64(9.999999977218428)),
 ('About a Boy', np.float64(9.999999976569251)),
 ('High Fidelity', np.float64(9.999999976569251)),
 ('Little Altars Everywhere: A Novel', np.float64(9.999999975304322)),
 ('Harry Potter and the Goblet of Fire (Book 4)',
  np.float64(9.999999974940073))]